<a href="https://colab.research.google.com/github/SAfsana/Data-Mining-Case-Studies/blob/main/Data_Mining_Case_Studies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Mining Case Studies – Google Colab Notebook
1. **Grocery Product Recommendation** – Apriori vs FP-Growth vs Eclat (support, confidence, lift, runtime, memory, confusion matrix)
2. **Sentiment Analysis of Delivery Reviews** – Logistic Regression vs Linear SVM vs Multinomial NB
3. **Lung Cancer Risk Prediction** – Decision Tree vs Random Forest vs Logistic Regression vs SVM

**How to use:** Runtime → Run all. Each case study asks you to upload its CSV file. Check the column names in the *Settings* cell of each case study and edit them if your CSV uses different names.
All numbers are computed from your data, so use these results in your slides.

In [ ]:
!pip -q install mlxtend
import re, time, random, tracemalloc, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
warnings.filterwarnings('ignore'); sns.set_style('whitegrid')

def get_file(prompt):
    from google.colab import files
    print(prompt); up = files.upload(); return next(iter(up))

def scores(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    return dict(Accuracy=accuracy_score(y, p), Precision=pr, Recall=rc, F1=f1)

def show_cm(cm, labels, title, ax):
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

## Case Study 1 – Grocery Product Recommendation (Apriori, FP-Growth, Eclat)

In [ ]:
# ---- Settings (edit if your column names differ) ----
MEMBER, DATE, ITEM = 'Member_number', 'Date', 'itemDescription'
MIN_SUPPORT = 0.002     # minimum support
MIN_CONF    = 0.05      # minimum confidence
TOP_N       = 5         # recommendations per basket
N_TEST      = 500       # test baskets used for evaluation

f1 = get_file('Upload the groceries CSV (e.g. Groceries_dataset.csv)')
df = pd.read_csv(f1); print(df.shape); df.head()

In [ ]:
# A basket = one customer on one date; keep the order of purchase
baskets = df.groupby([MEMBER, DATE])[ITEM].apply(lambda s: list(dict.fromkeys(s))).tolist()
baskets = [b for b in baskets if len(b) >= 2]
train, test = train_test_split(baskets, test_size=0.2, random_state=42)
items = sorted({i for b in baskets for i in b}); idx = {i: k for k, i in enumerate(items)}
print(f'transactions: {len(baskets):,} | unique products: {len(items)} | train/test: {len(train):,}/{len(test):,}')

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

te = TransactionEncoder()
X = pd.DataFrame(te.fit(baskets).transform(train), columns=te.columns_)

def eclat(transactions, min_support):
    # Eclat: vertical layout (item -> set of transaction ids), frequent itemsets by intersecting tid-sets
    n = len(transactions); min_cnt = int(np.ceil(min_support * n)); tid = {}
    for i, t in enumerate(transactions):
        for it in t: tid.setdefault(it, set()).add(i)
    out = []
    def rec(prefix, cands):
        for i, (it, tids) in enumerate(cands):
            new = prefix + (it,); out.append((frozenset(new), len(tids) / n)); nxt = []
            for it2, t2 in cands[i + 1:]:
                inter = tids & t2
                if len(inter) >= min_cnt: nxt.append((it2, inter))
            if nxt: rec(new, nxt)
    start = sorted([(k, v) for k, v in tid.items() if len(v) >= min_cnt], key=lambda x: len(x[1]))
    rec((), start)
    return pd.DataFrame(out, columns=['itemsets', 'support'])

def make_rules(fi):
    # rules with ONE item on the right side: support, confidence, lift
    sup = {frozenset(s): v for s, v in zip(fi['itemsets'], fi['support'])}; rows = []
    for s, v in sup.items():
        if len(s) < 2: continue
        for c in s:
            a = s - {c}
            if a in sup and frozenset([c]) in sup:
                conf = v / sup[a]; lift = conf / sup[frozenset([c])]
                if conf >= MIN_CONF and lift > 1: rows.append((a, c, v, conf, lift))
    return pd.DataFrame(rows, columns=['antecedent', 'consequent', 'support', 'confidence', 'lift'])

ALGOS = {'Apriori': lambda: apriori(X, min_support=MIN_SUPPORT, use_colnames=True),
         'FP-Growth': lambda: fpgrowth(X, min_support=MIN_SUPPORT, use_colnames=True),
         'Eclat': lambda: eclat(train, MIN_SUPPORT)}
res = {}
for name, fn in ALGOS.items():
    tracemalloc.start(); t0 = time.perf_counter(); fi = fn(); sec = time.perf_counter() - t0
    mem = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
    fi['itemsets'] = fi['itemsets'].apply(frozenset); rules = make_rules(fi)
    res[name] = dict(fi=fi, rules=rules, sec=sec, mem=mem)
    print(f'{name}: {len(fi)} itemsets, {len(rules)} rules, {sec:.2f}s, {mem:.1f} MB')

In [ ]:
# Table 1: support, confidence, lift per algorithm
t1 = pd.DataFrame({n: dict(Rules=len(r['rules']), Avg_Support=r['rules'].support.mean(),
        Avg_Confidence=r['rules'].confidence.mean(), Avg_Lift=r['rules'].lift.mean()) for n, r in res.items()}).T
# Table 2: execution time and memory
t2 = pd.DataFrame({n: {'Execution time (s)': r['sec'], 'Memory usage (MB)': r['mem']} for n, r in res.items()}).T
display(t1.round(4)); display(t2.round(3))
top = res['Apriori']['rules'].sort_values('lift', ascending=False).head(10).copy()
top['rule'] = top.antecedent.apply(lambda a: ' + '.join(sorted(a))) + ' -> ' + top.consequent
display(top[['rule', 'support', 'confidence', 'lift']].round(4))

In [ ]:
# Recommendation evaluation: hide the last product of each test basket, recommend top-N
def make_rec(rules):
    R = list(zip(rules.antecedent, rules.consequent, rules.confidence))
    def rec(basket):
        b = set(basket); sc = {}
        for a, c, conf in R:
            if a <= b and c not in b: sc[c] = max(sc.get(c, 0), conf)
        return [i for i, _ in sorted(sc.items(), key=lambda x: -x[1])[:TOP_N]]
    return rec

random.seed(42); test_b = random.sample([b for b in test if len(b) >= 2], min(N_TEST, len(test)))
ev = {}
for name, r in res.items():
    rec = make_rec(r['rules']); yt, yp = [], []
    for b in test_b:
        t = np.zeros(len(items), int); t[idx[b[-1]]] = 1
        p = np.zeros(len(items), int)
        for i in rec(b[:-1]): p[idx[i]] = 1
        yt.append(t); yp.append(p)
    yt, yp = np.concatenate(yt), np.concatenate(yp); cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn_, tp = cm.ravel()
    pr = tp / (tp + fp) if tp + fp else 0; rc = tp / (tp + fn_)
    ev[name] = dict(cm=cm, Accuracy=(tn + tp) / cm.sum(), Precision=pr, Recall=rc, F1=2 * pr * rc / (pr + rc) if pr + rc else 0)
t3 = pd.DataFrame({n: {k: v for k, v in e.items() if k != 'cm'} for n, e in ev.items()}).T
print(f'{len(test_b)} test baskets x {len(items)} products = {len(test_b) * len(items):,} checks'); display((t3 * 100).round(2))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
names = list(res)
for a, col, ttl in zip(ax[0], ['Execution time (s)', 'Memory usage (MB)'], ['Execution time (s)', 'Memory usage (MB)']):
    a.bar(names, t2[col], color=['#f5a623', '#2f80ed', '#2a9d6f']); a.set_title(ttl)
ax[0, 2].bar(names, t1.Avg_Lift, color='#8fb6dd'); ax[0, 2].set_title('Average lift'); ax[0, 2].set_ylim(1, None)
for a, n in zip(ax[1], names): show_cm(ev[n]['cm'], ['Not rec.', 'Recommended'], f'{n} confusion matrix', a)
plt.tight_layout(); plt.show()
top.set_index('rule')[['support', 'confidence', 'lift']].plot(kind='barh', subplots=True, layout=(1, 3), figsize=(16, 4), legend=False, sharey=True)
plt.suptitle('Top 10 rules by lift (Apriori)'); plt.tight_layout(); plt.show()

## Case Study 2 – Sentiment Analysis of Delivery Agent Reviews

In [ ]:
# ---- Settings ----
TEXT_COL, RATING_COL = 'Review', 'Rating'     # change to your column names
f2 = get_file('Upload the Zomato / delivery reviews CSV')
rv = pd.read_csv(f2); print(rv.columns.tolist(), rv.shape)
rv['rating'] = pd.to_numeric(rv[RATING_COL], errors='coerce'); rv = rv.dropna(subset=[TEXT_COL, 'rating'])
rv['label'] = np.where(rv.rating <= 2.0, 'Negative', np.where(rv.rating <= 3.5, 'Neutral', 'Positive'))
def clean(t):
    w = [x for x in re.sub('[^a-z ]', ' ', str(t).lower()).split() if len(x) >= 3]
    return ' '.join(w) if len(w) >= 3 else None
rv['clean'] = rv[TEXT_COL].apply(clean); rv = rv.dropna(subset=['clean'])
print(rv.label.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

L2 = ['Negative', 'Neutral', 'Positive']
Xtr, Xte, ytr, yte = train_test_split(rv.clean, rv.label, test_size=0.2, random_state=42, stratify=rv.label)
tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
Xtr_v, Xte_v = tf.fit_transform(Xtr), tf.transform(Xte)
M2 = {'Logistic Regression': LogisticRegression(max_iter=1000), 'Linear SVM': LinearSVC(), 'Multinomial NB': MultinomialNB()}
r2, p2 = {}, {}
for n, m in M2.items():
    p2[n] = m.fit(Xtr_v, ytr).predict(Xte_v); r2[n] = scores(yte, p2[n])
t4 = pd.DataFrame(r2).T; display((t4 * 100).round(2))
best2 = t4.F1.idxmax(); print('Best model:', best2); print(classification_report(yte, p2[best2], digits=3))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5)); (t4 * 100).plot(kind='bar', ax=ax[0], rot=0, ylim=(60, 100)); ax[0].set_title('Scores by model (%)')
show_cm(confusion_matrix(yte, p2[best2], labels=L2), L2, f'{best2} confusion matrix', ax[1]); plt.tight_layout(); plt.show()

## Case Study 3 – Lung Cancer Risk Prediction (Decision Tree, Random Forest, Logistic Regression, SVM)

In [ ]:
# ---- Settings ----
TARGET = 'Level'          # Low / Medium / High
f3 = get_file('Upload the lung cancer CSV')
lc = pd.read_csv(f3); lc = lc.drop(columns=[c for c in lc.columns if c.lower() in ('index', 'patient id', 'patient_id')])
print(lc.shape); print(lc[TARGET].value_counts())

Upload the lung cancer CSV


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

L3 = ['Low', 'Medium', 'High']
X3, y3 = lc.drop(columns=[TARGET]), lc[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)
M3 = {'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
      'Decision Tree': DecisionTreeClassifier(random_state=42),
      'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
      'SVM': make_pipeline(StandardScaler(), SVC())}
r3, p3 = {}, {}
for n, m in M3.items():
    p3[n] = m.fit(Xtr, ytr).predict(Xte); r3[n] = scores(yte, p3[n])
t5 = pd.DataFrame(r3).T.sort_values('F1', ascending=False); display((t5 * 100).round(2))
print('Best model:', t5.F1.idxmax(), '| train/test:', len(Xtr), '/', len(Xte))
fig, ax = plt.subplots(2, 3, figsize=(16, 9)); (t5 * 100).plot(kind='bar', ax=ax[0, 0], rot=20, ylim=(50, 100)); ax[0, 0].set_title('Scores by model (%)')
for a, n in zip(list(ax.flat)[1:], M3): show_cm(confusion_matrix(yte, p3[n], labels=L3), L3, f'{n} confusion matrix', a)
plt.tight_layout(); plt.show()
rf = M3['Random Forest']; imp = pd.Series(rf.feature_importances_, index=X3.columns).sort_values(ascending=False)
print('Top Random Forest features (a score near 100% can mean a feature gives away the risk level):'); display(imp.head(8).round(3))